In [1]:
import os, pathlib, subprocess, glob, re, time, shutil
BASE = pathlib.Path("/home/lobarevadv/projects/mrt/nnunet_fcd")
VENV = pathlib.Path("/home/lobarevadv/projects/mrt/.venv/bin")
os.environ["nnUNet_raw"]          = str(BASE / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(BASE / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(BASE / "nnUNet_results")
JOBS_CV = BASE / "jobs_cv"; JOBS_CV.mkdir(exist_ok=True)
PL = "nnUNetResEncUNetMPlans"
RES504 = pathlib.Path(os.environ["nnUNet_results"]) / "Dataset504_FCD_BonnIDEAS"
JOBS = [("0", "0", "nnUNetTrainer_FCD"),        # GPU0: oversample 0.33
        ("1", "0", "nnUNetTrainer_FCD_os05")]   # GPU1: oversample 0.5

def tail(path, n=5):
    p = pathlib.Path(path)
    print("".join(p.read_text(errors="ignore").splitlines(True)[-n:]) if p.exists() else f"(нет {p.name})")

for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    print(k, pathlib.Path(os.environ[k]).exists())

nnUNet_raw True
nnUNet_preprocessed True
nnUNet_results True


In [17]:
# остановить runner и nnU-Net (в том числе замороженные паузой)
subprocess.run("pkill -f jobs_cv/runner.py", shell=True)
subprocess.run("pkill -CONT -f nnUNetv2_train; pkill -9 -f 'nnUNetv2_train 504'", shell=True)
time.sleep(5)

# удалить результаты ТОЛЬКО этих двух тренеров на 504
for gpu, folds, tr in JOBS:
    d = RES504 / f"{tr}__{PL}__3d_fullres"
    if d.exists():
        shutil.rmtree(d); print("удалено:", d.name)

# очистить логи jobs_cv (runner.py остаётся)
for p in list(JOBS_CV.glob("*.log")) + list(JOBS_CV.glob("*.out")) + list(JOBS_CV.glob("*.nohup")):
    p.unlink()

print(subprocess.run("pgrep -af 'runner.py|nnUNetv2_train'", shell=True, capture_output=True, text=True).stdout or "ничего не запущено ✔")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv"], capture_output=True, text=True).stdout)

удалено: nnUNetTrainer_FCD__nnUNetResEncUNetMPlans__3d_fullres
удалено: nnUNetTrainer_FCD_os05__nnUNetResEncUNetMPlans__3d_fullres
3758330 /bin/sh -c pgrep -af 'runner.py|nnUNetv2_train'

index, memory.used [MiB]
0, 3 MiB
1, 0 MiB
2, 0 MiB



In [18]:
%%writefile /home/lobarevadv/projects/mrt/.venv/lib/python3.12/site-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer_FCD.py
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_250epochs


class nnUNetTrainer_FCD(nnUNetTrainer_250epochs):
    # 250 эпох, oversample 0.33 (по умолчанию)
    def on_train_start(self):
        super().on_train_start()
        self.save_every = 10


class nnUNetTrainer_FCD_os05(nnUNetTrainer_250epochs):
    # 250 эпох, oversample 0.5
    def initialize(self):
        self.oversample_foreground_percent = 0.5
        super().initialize()

    def on_train_start(self):
        super().on_train_start()
        self.save_every = 10

Overwriting /home/lobarevadv/projects/mrt/.venv/lib/python3.12/site-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer_FCD.py


In [19]:
f = pathlib.Path("/home/lobarevadv/projects/mrt/.venv/lib/python3.12/site-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer_FCD.py")
print("os05 в файле:", "nnUNetTrainer_FCD_os05" in f.read_text())
for t in ("nnUNetTrainer_FCD", "nnUNetTrainer_FCD_os05"):
    r = subprocess.run([str(VENV / "python"), "-c",
        f"from nnunetv2.utilities.find_objects import recursive_find_trainer_class_by_name as f; print(f('{t}'))"],
        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-300:])

os05 в файле: True
<class 'nnunetv2.training.nnUNetTrainer.nnUNetTrainer_FCD.nnUNetTrainer_FCD'>
<class 'nnunetv2.training.nnUNetTrainer.nnUNetTrainer_FCD.nnUNetTrainer_FCD_os05'>


In [20]:
%%writefile /home/lobarevadv/projects/mrt/nnunet_fcd/jobs_cv/runner.py
import subprocess, time, sys, os, signal, pathlib, datetime
GPU, FOLDS, TR = sys.argv[1], sys.argv[2].split(","), sys.argv[3]
assert GPU in ("0", "1"), "GPU 2 запрещена"

BASE = pathlib.Path("/home/lobarevadv/projects/mrt/nnunet_fcd")
VENV = pathlib.Path("/home/lobarevadv/projects/mrt/.venv/bin")
os.environ["nnUNet_raw"]          = str(BASE / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(BASE / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(BASE / "nnUNet_results")
DS, PL = "504", "nnUNetResEncUNetMPlans"
RES = BASE / "nnUNet_results" / "Dataset504_FCD_BonnIDEAS" / f"{TR}__{PL}__3d_fullres"
LOG = BASE / "jobs_cv"
T_HI, T_LO = 85, 78

def temp():
    try:
        out = subprocess.run(["nvidia-smi", "-i", GPU, "--query-gpu=temperature.gpu",
                              "--format=csv,noheader"], capture_output=True, text=True, timeout=30).stdout
        return int(out.strip())
    except Exception:
        return 0

def log(m):
    with open(LOG / f"gpu{GPU}_{TR}.log", "a") as f:
        f.write(f"{datetime.datetime.now():%m-%d %H:%M} {m}\n")

def stop_all(p):
    # корректно завершить, даже если процесс на паузе
    try:
        os.killpg(p.pid, signal.SIGCONT); os.killpg(p.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

log(f"runner старт: folds={FOLDS}")
for f in FOLDS:
    fd = RES / f"fold_{f}"
    for attempt in range(4):
        if (fd / "checkpoint_final.pth").exists():
            log(f"fold {f} готов"); break
        cmd = [str(VENV / "nnUNetv2_train"), DS, "3d_fullres", f, "-tr", TR, "-p", PL, "--npz"]
        if (fd / "checkpoint_latest.pth").exists(): cmd.append("--c")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU, nnUNet_n_proc_DA="6",
                   PATH=f"{VENV}:{os.environ.get('PATH','')}")
        log(f"start fold {f}, попытка {attempt}: {' '.join(cmd[1:])}")
        with open(LOG / f"fold{f}_{TR}.out", "a") as out:
            p = subprocess.Popen(["nice", "-n", "10"] + cmd, env=env, stdout=out,
                                 stderr=subprocess.STDOUT, start_new_session=True)
        signal.signal(signal.SIGTERM, lambda *a: (stop_all(p), sys.exit(0)))
        paused = False
        while p.poll() is None:
            t = temp()
            if t >= T_HI and not paused:
                os.killpg(p.pid, signal.SIGSTOP); paused = True; log(f"ПАУЗА {t}°C")
            elif t <= T_LO and paused:
                os.killpg(p.pid, signal.SIGCONT); paused = False; log(f"продолжение {t}°C")
            time.sleep(30)
        log(f"fold {f} вышел, код {p.returncode}")
        if p.returncode == 0 and (fd / "checkpoint_final.pth").exists(): break
        time.sleep(120)
log("очередь завершена")

Overwriting /home/lobarevadv/projects/mrt/nnunet_fcd/jobs_cv/runner.py


In [21]:
running = subprocess.run(["pgrep", "-af", "jobs_cv/runner.py"], capture_output=True, text=True).stdout
for gpu, folds, tr in JOBS:
    if f"runner.py {gpu} " in running:
        print(f"GPU{gpu}: уже работает, не запускаю второй раз"); continue
    subprocess.Popen(
        f"nohup {VENV}/python {JOBS_CV}/runner.py {gpu} {folds} {tr} > {JOBS_CV}/runner{gpu}.nohup 2>&1 &",
        shell=True, start_new_session=True)
    print(f"GPU{gpu}: запущен {tr}, фолды {folds}")

GPU0: запущен nnUNetTrainer_FCD, фолды 0
GPU1: запущен nnUNetTrainer_FCD_os05, фолды 0


In [3]:
print(subprocess.run(["nvidia-smi", "--query-gpu=index,temperature.gpu,utilization.gpu,memory.used",
                      "--format=csv"], capture_output=True, text=True).stdout)
for gpu, folds, tr in JOBS:
    print(f"===== GPU{gpu} {tr}")
    tail(JOBS_CV / f"gpu{gpu}_{tr}.log", 3)
    lg = sorted(glob.glob(f"{RES504}/{tr}__{PL}__3d_fullres/fold_0/training_log_*.txt"))
    if lg:
        txt = open(lg[-1]).read()
        ep = re.findall(r"Epoch (\d+)", txt); et = re.findall(r"Epoch time: ([\d.]+)", txt)
        ps = re.findall(r"Pseudo dice \[(.*?)\]", txt)
        if ep: print(f"эпоха {ep[-1]}/250, время эпохи {et[-1] if et else '?'} с, pseudo dice {ps[-1] if ps else '?'}")
    tail(JOBS_CV / f"fold0_{tr}.out", 3)

index, temperature.gpu, utilization.gpu [%], memory.used [MiB]
0, 62, 2 %, 0 MiB
1, 61, 2 %, 0 MiB
2, 60, 0 %, 0 MiB

===== GPU0 nnUNetTrainer_FCD
11-09 10:17 продолжение 69°C
11-09 10:17 ПАУЗА 86°C
11-09 10:18 продолжение 69°C

эпоха 249/250, время эпохи 251.42 с, pseudo dice np.float32(0.3517)
2026-11-09 10:18:35.520944: FCD_133, shape torch.Size([2, 320, 320, 200]), rank 0
/usr/lib/python3.12/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 6 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '

===== GPU1 nnUNetTrainer_FCD_os05
11-09 09:54 продолжение 68°C
11-09 09:55 fold 0 вышел, код 0
11-09 09:55 очередь завершена

эпоха 249/250, время эпохи 252.04 с, pseudo dice np.float32(0.3218)
2026-11-09 09:54:51.394887: IDEAS_4097, shape torch.Size([2, 214, 226, 191]), rank 0
2026-11-09 09:55:07.532170: Validation complete
2026-11-09 09:55:07.532289: Mean Validation Dice:  0.04592837252258316



In [6]:
%%writefile /home/lobarevadv/projects/mrt/nnunet_fcd/jobs_cv/val_runner.py
import subprocess, time, os, signal, datetime, pathlib, sys
GPU, TR = sys.argv[1], sys.argv[2]
assert GPU in ("0", "1")
BASE = pathlib.Path("/home/lobarevadv/projects/mrt/nnunet_fcd"); VENV = "/home/lobarevadv/projects/mrt/.venv/bin"
for k in ("raw", "preprocessed", "results"): os.environ[f"nnUNet_{k}"] = str(BASE / f"nnUNet_{k}")
LOG = BASE / "jobs_cv" / f"val{GPU}.log"
def log(m): open(LOG, "a").write(f"{datetime.datetime.now():%m-%d %H:%M} {m}\n")
def temp():
    try: return int(subprocess.run(["nvidia-smi", "-i", GPU, "--query-gpu=temperature.gpu", "--format=csv,noheader"],
                                   capture_output=True, text=True, timeout=30).stdout.strip())
    except Exception: return 0
cmd = ["nice", "-n", "10", f"{VENV}/nnUNetv2_train", "504", "3d_fullres", "0", "-tr", TR,
       "-p", "nnUNetResEncUNetMPlans", "--val", "--npz"]
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU, PATH=f"{VENV}:{os.environ['PATH']}")
log("старт валидации")
p = subprocess.Popen(cmd, env=env, stdout=open(BASE / "jobs_cv" / f"val{GPU}.out", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
paused = False
while p.poll() is None:
    t = temp()
    if t >= 85 and not paused: os.killpg(p.pid, signal.SIGSTOP); paused = True; log(f"ПАУЗА {t}°C")
    elif t <= 78 and paused:   os.killpg(p.pid, signal.SIGCONT); paused = False; log(f"продолжение {t}°C")
    time.sleep(20)
log(f"валидация завершена, код {p.returncode}")

Writing /home/lobarevadv/projects/mrt/nnunet_fcd/jobs_cv/val_runner.py


In [7]:
subprocess.Popen(f"nohup {VENV}/python {JOBS_CV}/val_runner.py 0 nnUNetTrainer_FCD > /dev/null 2>&1 &",
                 shell=True, start_new_session=True)
print("валидация GPU0 запущена")

валидация GPU0 запущена


In [2]:
tail(JOBS_CV / "val0.log", 3)
tail(JOBS_CV / "val0.out", 3)
v = RES504 / f"nnUNetTrainer_FCD__{PL}__3d_fullres" / "fold_0" / "validation"
print("npz:", len(list(v.glob("*.npz"))), "из 38;  summary.json:", (v / "summary.json").exists())

11-09 12:14 ПАУЗА 85°C
11-09 12:14 продолжение 71°C
11-09 12:15 валидация завершена, код 0

2026-11-09 12:14:41.854301: IDEAS_4097, shape torch.Size([2, 214, 226, 191]), rank 0
2026-11-09 12:14:57.070455: Validation complete
2026-11-09 12:14:57.070565: Mean Validation Dice:  0.04012251892275462

npz: 38 из 38;  summary.json: True


In [3]:
import numpy as np, nibabel as nib
from scipy import ndimage
LAB = pathlib.Path(os.environ["nnUNet_raw"]) / "Dataset504_FCD_BonnIDEAS" / "labelsTr"

def load_val(tr):
    out = []
    for npz in sorted((RES504 / f"{tr}__{PL}__3d_fullres" / "fold_0" / "validation").glob("*.npz")):
        cid = npz.stem
        prob = np.load(npz)["probabilities"][1].astype(np.float16)
        gt = np.asanyarray(nib.load(str(LAB / f"{cid}.nii.gz")).dataobj).astype(np.uint8).transpose(2, 1, 0)
        assert prob.shape == gt.shape, f"{cid}: {prob.shape} vs {gt.shape}"
        out.append((cid, prob, gt))
    return out

def score(cases, t, ms):
    r = dict(bonn_tp=0, bonn_n=0, ideas_tp=0, ideas_n=0, ctrl_ok=0, ctrl_n=0, fp=0)
    for cid, prob, gt in cases:
        lab, n = ndimage.label(prob > t)
        sizes = ndimage.sum(np.ones(lab.shape, np.uint8), lab, range(1, n + 1)) if n else []
        keep = [i + 1 for i, s in enumerate(sizes) if s >= ms]
        target = gt > 0                                   # очаг (1) или резекция (2)
        hit = [k for k in keep if (target & (lab == k)).any()]
        r["fp"] += len(keep) - len(hit)
        if (gt == 1).any():   r["bonn_n"] += 1;  r["bonn_tp"] += bool(hit)
        elif (gt == 2).any(): r["ideas_n"] += 1; r["ideas_tp"] += bool(hit)
        else:                 r["ctrl_n"] += 1;  r["ctrl_ok"] += not keep
    r["fp_per_case"] = r["fp"] / len(cases)
    return r

results = {}
for tr in ("nnUNetTrainer_FCD", "nnUNetTrainer_FCD_os05"):
    cases = load_val(tr)
    print(f"\n===== {tr}: {len(cases)} кейсов")
    rows = []
    for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
        for ms in (20, 50, 100, 200):
            s = score(cases, t, ms)
            rec  = s["bonn_tp"] / max(s["bonn_n"], 1)
            spec = s["ctrl_ok"] / max(s["ctrl_n"], 1)
            rows.append((t, ms, rec, spec, s))
            print(f"t={t:.1f} min={ms:3d} | Bonn recall {s['bonn_tp']}/{s['bonn_n']}={rec:.2f} | "
                  f"IDEAS {s['ideas_tp']}/{s['ideas_n']} | spec {s['ctrl_ok']}/{s['ctrl_n']}={spec:.2f} | FP/кейс {s['fp_per_case']:.2f}")
    ok = [x for x in rows if x[3] >= 0.85] or rows
    best = max(ok, key=lambda x: (x[2], x[4]["ideas_tp"], -x[4]["fp_per_case"]))
    results[tr] = best
    print(f">>> ЛУЧШЕЕ: порог {best[0]}, мин. размер {best[1]}, recall {best[2]:.2f}, spec {best[3]:.2f}, FP/кейс {best[4]['fp_per_case']:.2f}")
    del cases

print("\n===== ИТОГ")
for tr, b in results.items():
    print(f"{tr:28s} порог {b[0]} | min {b[1]} | Bonn recall {b[2]:.2f} | spec {b[3]:.2f} | IDEAS {b[4]['ideas_tp']}/{b[4]['ideas_n']} | FP/кейс {b[4]['fp_per_case']:.2f}")


===== nnUNetTrainer_FCD: 38 кейсов
t=0.2 min= 20 | Bonn recall 3/12=0.25 | IDEAS 0/1 | spec 11/25=0.44 | FP/кейс 1.32
t=0.2 min= 50 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 11/25=0.44 | FP/кейс 1.18
t=0.2 min=100 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 12/25=0.48 | FP/кейс 0.95
t=0.2 min=200 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 18/25=0.72 | FP/кейс 0.55
t=0.3 min= 20 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 11/25=0.44 | FP/кейс 1.29
t=0.3 min= 50 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 12/25=0.48 | FP/кейс 1.03
t=0.3 min=100 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 18/25=0.72 | FP/кейс 0.68
t=0.3 min=200 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 18/25=0.72 | FP/кейс 0.55
t=0.4 min= 20 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 11/25=0.44 | FP/кейс 1.16
t=0.4 min= 50 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 16/25=0.64 | FP/кейс 0.84
t=0.4 min=100 | Bonn recall 2/12=0.17 | IDEAS 0/1 | spec 18/25=0.72 | FP/кейс 0.63
t=0.4 min=200 | Bonn recall 2/12=0.17 | IDEAS 0/1 |